导入工具

In [1]:
import sys
import os
import json
import hashlib
import winreg

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen

if sys.version_info[:3] != (3, 13, 16):
    raise RuntimeError("请切换到项目 .venv 的 Python 3.13.16 内核。")

print("Python 版本：", sys.version.split()[0])

Python 版本： 3.13.16


定位桌面项目，检查是否使用正确的虚拟环境

In [2]:
# 读取 Windows 配置的真实桌面路径，支持 OneDrive 或其他重定向。
registry_path = (
    r"Software\Microsoft\Windows\CurrentVersion"
    r"\Explorer\User Shell Folders"
)

with winreg.OpenKey(winreg.HKEY_CURRENT_USER, registry_path) as key:
    configured_desktop, _ = winreg.QueryValueEx(key, "Desktop")

desktop = Path(os.path.expandvars(configured_desktop))

if not desktop.is_absolute() or not desktop.is_dir():
    raise RuntimeError("Windows 桌面路径不可用，请先核实路径。")

# 只使用已有项目，不在错误位置重新创建项目。
PROJECT_ROOT = desktop / "tft-balance-simulator"

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("未找到已初始化的桌面项目。")

expected_venv = (PROJECT_ROOT / ".venv").resolve()

if Path(sys.prefix).resolve() != expected_venv:
    raise RuntimeError("当前内核不是这个项目的 .venv，请切换内核。")

print("项目定位成功：", PROJECT_ROOT.name)
print("虚拟环境检查通过")

项目定位成功： tft-balance-simulator
虚拟环境检查通过


下载 TFT Set 10、Patch 14.1 的真实游戏数据，并检查下载是否成功

In [ ]:
PATCH = "14.1"
SET_NUMBER = 10
LANGUAGE = "en_us"

SOURCE_URL = (
    f"https://raw.communitydragon.org/"
    f"{PATCH}/cdragon/tft/{LANGUAGE}.json"
)

request = Request(
    SOURCE_URL,
    headers={"User-Agent": "tft-balance-simulator-source-audit/0.1"},
)

with urlopen(request, timeout=60) as response:
    http_status = response.status

    max_bytes = 25 * 1024 * 1024
    raw_bytes = response.read(max_bytes + 1)

    # 仅保存可公开的来源响应头
    source_headers = {
        "content_type": response.headers.get("Content-Type"),
        "last_modified": response.headers.get("Last-Modified"),
        "etag": response.headers.get("ETag"),
    }

if http_status != 200:
    raise RuntimeError(f"请求未成功：HTTP {http_status}")

if len(raw_bytes) > max_bytes:
    raise RuntimeError("响应超过 25 MiB，请先检查来源。")

# 保留原始字节，后续保存时不重新格式化上游文件。
retrieved_at = datetime.now(timezone.utc)
source_sha256 = hashlib.sha256(raw_bytes).hexdigest()

print("HTTP 状态：", http_status)
print("文件字节数：", len(raw_bytes))
print("SHA-256：", source_sha256)

HTTP 状态： 200
文件字节数： 6414768
SHA-256： ae6d4bde7f591194b599caa120ca36ccc592bb5d9942ae703a47bd0cd0f7cb56


将下载的JSON数据转换成Python可以操作的数据结构，然后找到Set10的英雄和基础属性

In [ ]:
# 将原始 JSON 转换成 Python 字典，不修改上游内容。
payload = json.loads(raw_bytes)

if not isinstance(payload, dict):
    raise TypeError("JSON 顶层不是预期的字典结构。")

sets = payload.get("sets")

if not isinstance(sets, dict):
    raise ValueError("缺少 sets 字典，请检查上游结构。")

selected_set = sets.get(str(SET_NUMBER))

if not isinstance(selected_set, dict):
    raise ValueError("没有找到 Set 10。")

champions = selected_set.get("champions")    # 从S10中读取英雄列表

if not isinstance(champions, list) or not champions:
    raise ValueError("Set 10 的 champions 列表不存在或为空。")

print("顶层字段：", sorted(payload.keys()))
print("文件中的 Set：", sorted(sets.keys()))
print("选定 Set 名称：", selected_set.get("name"))
print("原始单位记录数：", len(champions))

# 按稳定的内部 ID 查找样例，不依赖列表第一条恰好是可玩英雄。
sample = next(
    (unit for unit in champions if unit.get("apiName") == "TFT10_Annie"),
    None,
)

if sample is None:
    raise ValueError("没有找到 Annie 样例，请先核实数据版本。")

print("\n样例英雄：", sample.get("name"))
print("费用：", sample.get("cost"))
print("原始属性：")
print(json.dumps(sample.get("stats"), ensure_ascii=False, indent=2))

顶层字段： ['items', 'setData', 'sets']
文件中的 Set： ['1', '10', '9']
选定 Set 名称： Set10
原始单位记录数： 76

样例英雄： Annie
费用： 1
原始属性：
{
  "armor": 20.0,
  "attackSpeed": 0.6499999761581421,
  "critChance": 0.25,
  "critMultiplier": 1.399999976158142,
  "damage": 40.0,
  "hp": 450.0,
  "initialMana": 0,
  "magicResist": 20.0,
  "mana": 50.0,
  "range": 5.0
}


把前面下载到Python内存中的TFT原始数据永久保存到项目，同时建立数据来源记录，确保以后能够复现实验

In [5]:
# 使用 UTC 时间和哈希前缀创建独立快照目录，避免覆盖历史下载。
snapshot_name = (
    retrieved_at.strftime("%Y%m%dT%H%M%S%fZ")
    + "_"
    + source_sha256[:12]
)

snapshot_dir = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / f"set{SET_NUMBER}_patch{PATCH.replace('.', '_')}"
    / snapshot_name
)

snapshot_dir.mkdir(parents=True, exist_ok=False)

raw_path = snapshot_dir / f"{LANGUAGE}.json"
manifest_path = snapshot_dir / "manifest.json"

# xb 表示以二进制方式独占创建；文件已存在时直接报错。
with raw_path.open("xb") as file:
    file.write(raw_bytes)

# 元数据只记录项目相对路径，不保存本机用户名或绝对路径。
manifest = {
    "source_provider": "CommunityDragon",
    "source_url": SOURCE_URL,
    "patch_directory": PATCH,
    "set_number": SET_NUMBER,
    "set_name": selected_set.get("name"),
    "language": LANGUAGE,
    "retrieved_at_utc": retrieved_at.isoformat(),
    "sha256": source_sha256,
    "byte_length": len(raw_bytes),
    "raw_file": raw_path.relative_to(PROJECT_ROOT).as_posix(),
    "raw_entity_count": len(champions),
    "source_headers": source_headers,
    "hotfix_status": "not independently verified",
}

with manifest_path.open("x", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

# 回读文件，检查磁盘内容是否与下载内容一致。
saved_sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()

if saved_sha256 != source_sha256:
    raise RuntimeError("保存后的文件指纹不一致，请停止后续处理。")

print("快照目录：", snapshot_dir.relative_to(PROJECT_ROOT).as_posix())
print("原始文件与来源记录已保存")
print("文件完整性检查通过")

快照目录： data/raw/set10_patch14_1/20261027T211906309960Z_ae6d4bde7f59
原始文件与来源记录已保存
文件完整性检查通过
